In [2]:
# imports
import pandas as pd
import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from constant import DATA_DIR, RANDOM_STATE


2026-11-16 03:18:01.253578: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-16 03:18:01.377517: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-11-16 03:18:03.537499: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# Viewing the Data w/ Pandas

In [3]:
# loading the data
clf_df = pd.read_csv(f'{DATA_DIR}/draft.csv')

In [4]:
# looking at what the data looks like
clf_df.head(),clf_df.dtypes

(  position  longsnapper  height  weight   draftage  bench  forty  \
 0       OL            0   81.25   322.0  22.112329     19   5.40   
 1       OL            0   81.00   317.0  24.227397     16   5.46   
 2       OL            0   80.75   324.0  22.520548     18   5.53   
 3       OL            0   80.00   335.0  23.276712     17   0.00   
 4       OL            0   80.00   320.0  25.243836     22   0.00   
 
    fortyelectric  verticaljump  broadjump  threecone  shuttle  drafted  
 0          5.430         28.50     95.000      7.470    4.660        1  
 1          5.790         26.75     96.624      8.280    5.000        1  
 2          5.540         24.50     87.500      8.180    4.900        1  
 3          5.453         28.00     96.000      8.047    4.918        1  
 4          5.280         33.50     93.000      7.500    4.610        1  ,
 position             str
 longsnapper        int64
 height           float64
 weight           float64
 draftage         float64
 bench   

In [5]:
COLS = [
        "height", "weight", "draftage", "forty","bench",
        "fortyelectric", "verticaljump", "broadjump", "threecone", "shuttle"
    ]
def get_rows_with_zero(df,cols_to_remove):
    """A function that returns the dataframe that has 0 within the columns
    that do not contain a 0 by default(i.e.,height,weight,draftage,
    bench,forty,fortyelectric,verticaljump,broadjump,threecone,shuttle.).

    """

    cols = [c for c in COLS if c not in cols_to_remove]
    # for all the rows, for all the columns, we check if it equals 0
    # then return a true for that row else false(via .any).
    mask =(df[cols] == 0).any(axis=1)
    return df[mask]

get_rows_with_zero(clf_df,[]).count(),clf_df.count()
# well this is problematic since that a significant amount of the dataset. as in,
# 556/671 of them contain a 0, meaning thats missing data. so we cant remove them.
# thus, we can take the average for all rows and simply replace them
# HOWEVER, I have no clue how to do this for bench and it seems like its just two inputs,
# I am going to remove them.


    
    

(position         556
 longsnapper      556
 height           556
 weight           556
 draftage         556
 bench            556
 forty            556
 fortyelectric    556
 verticaljump     556
 broadjump        556
 threecone        556
 shuttle          556
 drafted          556
 dtype: int64,
 position         671
 longsnapper      671
 height           671
 weight           671
 draftage         671
 bench            671
 forty            671
 fortyelectric    671
 verticaljump     671
 broadjump        671
 threecone        671
 shuttle          671
 drafted          671
 dtype: int64)

In [6]:
def handle_zero_within_cols(df):
    """This function simply takes the average of the columns that do not have 0 by default.
    Then fills the 0 with the average, with the exception of bench, which those rows were removed.
    """
    # Mask momement to return a df where bench does not equal 0
    # essentially removing bench
    df = df[df["bench"] != 0]
    for col in COLS:
        avg = df[col].mean()
        df[col] = df[col].replace(0,avg)
    return df

clf_df = handle_zero_within_cols(clf_df)
clf_df
    

,position,longsnapper,height,weight,draftage,bench,forty,fortyelectric,verticaljump,broadjump,threecone,shuttle,drafted
0,OL,0,81.250,322.0,22.112329,19,5.400000,5.430,28.50,95.000,7.470000,4.660,1
1,OL,0,81.000,317.0,24.227397,16,5.460000,5.790,26.75,96.624,8.280000,5.000,1
2,OL,0,80.750,324.0,22.520548,18,5.530000,5.540,24.50,87.500,8.180000,4.900,1
3,OL,0,80.000,335.0,23.276712,17,3.431146,5.453,28.00,96.000,8.047000,4.918,1
4,OL,0,80.000,320.0,25.243836,22,3.431146,5.280,33.50,93.000,7.500000,4.610,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,RB,0,67.875,189.0,23.509589,15,4.780000,4.900,34.00,112.000,6.880000,3.970,0
666,WR,0,67.875,176.0,23.298630,20,4.310000,4.540,43.50,125.000,3.534859,4.030,1
668,RB,0,67.375,182.0,25.627397,6,3.431146,4.740,27.50,89.500,7.240000,4.350,0
669,K,0,67.000,165.0,21.753425,1,3.431146,4.979,30.00,102.500,7.560000,4.500,1


In [7]:
get_rows_with_zero(clf_df,[]).count(),clf_df.count()

(position         0
 longsnapper      0
 height           0
 weight           0
 draftage         0
 bench            0
 forty            0
 fortyelectric    0
 verticaljump     0
 broadjump        0
 threecone        0
 shuttle          0
 drafted          0
 dtype: int64,
 position         646
 longsnapper      646
 height           646
 weight           646
 draftage         646
 bench            646
 forty            646
 fortyelectric    646
 verticaljump     646
 broadjump        646
 threecone        646
 shuttle          646
 drafted          646
 dtype: int64)

In [8]:
# one hot encoding the position
def position_map_to_numerical(df):
    """A function that takes in a df. Grabs all the unique str within
    the `position` col. Then convert them into numerical values.

    It does not make sense to map each position label to a numerical value
    in acsending order because then we are saying one position might be better than
    the other. This is the differeniation between categorical and ordinal, as in,
    position is categorical, thus, we need a one hot encoding.
    see<https://towardsdatascience.com/encoding-categorical-variables-in-machine-learning-5644a43c123f/>
    Thus, we want to one hot encode it.
    """

    # TODO(shenukp1) Nov 15, 2026: might need a hash map of this later
    unique_arr = df["position"].unique()
    
    # see<https://www.geeksforgeeks.org/machine-learning/one-hot-encoding-from-a-pandas-column-containing-a-list/>
    one_hot = pd.get_dummies(df['position'].apply(pd.Series).stack()).groupby(level=0).sum()
    df_one_hot = one_hot.join(df.drop('position', axis=1))
    return df_one_hot
clf_df = position_map_to_numerical(clf_df)
clf_df.head()
    

,DB,DL,K,LB,OL,QB,RB,WR,longsnapper,height,weight,draftage,bench,forty,fortyelectric,verticaljump,broadjump,threecone,shuttle,drafted
0,0,0,0,0,1,0,0,0,0,81.25,322.0,22.112329,19,5.400000,5.430,28.50,95.000,7.470,4.660,1
1,0,0,0,0,1,0,0,0,0,81.00,317.0,24.227397,16,5.460000,5.790,26.75,96.624,8.280,5.000,1
2,0,0,0,0,1,0,0,0,0,80.75,324.0,22.520548,18,5.530000,5.540,24.50,87.500,8.180,4.900,1
3,0,0,0,0,1,0,0,0,0,80.00,335.0,23.276712,17,3.431146,5.453,28.00,96.000,8.047,4.918,1
4,0,0,0,0,1,0,0,0,0,80.00,320.0,25.243836,22,3.431146,5.280,33.50,93.000,7.500,4.610,1


In [9]:
# Now we need to split this into x(data) and prediction(y) variables
X, Y = clf_df.iloc[:,:-1],clf_df.iloc[:,-1]
X,Y

(     DB  DL  K  LB  OL  QB  RB  WR  longsnapper  height  weight   draftage  \
 0     0   0  0   0   1   0   0   0            0  81.250   322.0  22.112329   
 1     0   0  0   0   1   0   0   0            0  81.000   317.0  24.227397   
 2     0   0  0   0   1   0   0   0            0  80.750   324.0  22.520548   
 3     0   0  0   0   1   0   0   0            0  80.000   335.0  23.276712   
 4     0   0  0   0   1   0   0   0            0  80.000   320.0  25.243836   
 ..   ..  .. ..  ..  ..  ..  ..  ..          ...     ...     ...        ...   
 665   0   0  0   0   0   0   1   0            0  67.875   189.0  23.509589   
 666   0   0  0   0   0   0   0   1            0  67.875   176.0  23.298630   
 668   0   0  0   0   0   0   1   0            0  67.375   182.0  25.627397   
 669   0   0  1   0   0   0   0   0            0  67.000   165.0  21.753425   
 670   0   0  0   0   0   0   1   0            0  66.875   168.0  24.550685   
 
      bench     forty  fortyelectric  verticaljump

In [10]:
# The lectures mentioned that a 80/20 split is industry standard
X_train, X_test,y_train, y_test = train_test_split(X,Y,
                                   random_state=RANDOM_STATE, 
                                   test_size=0.20)


In [11]:
X_train.shape, X_test.shape,y_train.shape,y_test.shape

((516, 19), (130, 19), (516,), (130,))

In [12]:
def save_draft_test_train(X_train, X_test,y_train, y_test):
    """A function that concatentes the X with their y
    for the specific split(i.e., train or test) and saves them
    as csv.
    """
    # see<https://stackoverflow.com/questions/69658534/append-data-to-training-dataset-after-train-test-split>
    # see<https://pandas.pydata.org/docs/reference/api/pandas.concat.html> for default axis is 0
    # but we want to stack the columns
    train_df = pd.concat([X_train, y_train], axis=1)
    test_df =  pd.concat([X_test, y_test], axis=1)
    train_df.to_csv(f"{DATA_DIR}/draft_train.csv", index=False)
    test_df.to_csv(f"{DATA_DIR}/draft_test.csv", index=False)
    
save_draft_test_train(X_train, X_test,y_train, y_test)

In [22]:
# training time
# see <https://www.tensorflow.org/tutorials/keras/overfit_and_underfit>
# We need this because I am training around 500 epochs and I want to stop when
# we do not see any improvement within 50 epoch
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',         
    patience=50,                 
    restore_best_weights=True,   # apparently this takes the best model for the best val loss, see<https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping>
    verbose=1 # 
)
model = tf.keras.Sequential([
 tf.keras.layers.Dense(32, activation='relu'),
 tf.keras.layers.Dropout(0.2),
 tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam',
              loss='binary_crossentropy',# because we have a binary output
              metrics=['accuracy'])
model.fit(X_train, y_train, epochs=1000,callbacks=[early_stopping],validation_split=0.2, verbose=2)

Epoch 1/1000
13/13 - 1s - 61ms/step - accuracy: 0.5049 - loss: 12.6246 - val_accuracy: 0.6442 - val_loss: 3.1764
Epoch 2/1000
13/13 - 0s - 9ms/step - accuracy: 0.5388 - loss: 12.4955 - val_accuracy: 0.6635 - val_loss: 2.6034
Epoch 3/1000
13/13 - 0s - 8ms/step - accuracy: 0.5388 - loss: 11.2417 - val_accuracy: 0.6442 - val_loss: 2.1697
Epoch 4/1000
13/13 - 0s - 8ms/step - accuracy: 0.5097 - loss: 10.7911 - val_accuracy: 0.6442 - val_loss: 3.8094
Epoch 5/1000
13/13 - 0s - 8ms/step - accuracy: 0.5291 - loss: 10.0522 - val_accuracy: 0.6442 - val_loss: 3.5516
Epoch 6/1000
13/13 - 0s - 8ms/step - accuracy: 0.5461 - loss: 8.6892 - val_accuracy: 0.6442 - val_loss: 3.4219
Epoch 7/1000
13/13 - 0s - 8ms/step - accuracy: 0.5073 - loss: 9.0273 - val_accuracy: 0.6442 - val_loss: 2.7931
Epoch 8/1000
13/13 - 0s - 8ms/step - accuracy: 0.4854 - loss: 8.5902 - val_accuracy: 0.6538 - val_loss: 2.4678
Epoch 9/1000
13/13 - 0s - 8ms/step - accuracy: 0.5437 - loss: 6.2736 - val_accuracy: 0.6538 - val_loss: 2.

In [23]:
print("--Evaluate model--")
model_loss1, model_acc1 = model.evaluate(X_train,  y_train, verbose=2)
model_loss2, model_acc2 = model.evaluate(X_test,  y_test, verbose=2)
print(f"Train / Test Accuracy: {model_acc1*100:.1f}% / {model_acc2*100:.1f}%")

--Evaluate model--
17/17 - 0s - 3ms/step - accuracy: 0.6531 - loss: 0.5638
5/5 - 0s - 8ms/step - accuracy: 0.7231 - loss: 0.5814
Train / Test Accuracy: 65.3% / 72.3%


In [15]:
# checking the output of the model
print(X_test.iloc[[0]])
print(y_test.iloc[0])
prediction = model.predict(X_test.iloc[[0]])
print(prediction)

     DB  DL  K  LB  OL  QB  RB  WR  longsnapper  height  weight   draftage  \
657   0   0  0   0   0   0   1   0            0  68.625   210.0  24.775342   

     bench  forty  fortyelectric  verticaljump  broadjump  threecone  shuttle  
657     23   4.76           4.84          34.5     106.25       7.14     4.19  
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
[[0.54557854]]
